# 04 - Delay Distribution by Hour

This notebook generates a bar chart showing delay distribution by hour of day.

**Features:**
- Stacked bar chart with On Time, Minor, Major delays
- Highlights commute periods
- Interactive hover tooltips

**Output:** `static/plots/delay_by_hour.html`

In [1]:
import os
import sys
import sqlite3
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..','..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

DB_PATH = os.path.join(project_root, 'data', 'caltrain_lat_long.db')
GTFS_PATH = os.path.join(project_root, 'gtfs_data')
OUTPUT_DIR = os.path.join(project_root, 'static', 'plots')

print(f"Database exists: {os.path.exists(DB_PATH)}")

Database exists: True


## Configuration

In [2]:
# ====================
# CONFIGURATION
# ====================

# Date range (None = all data)
START_DATE = '2025-01-01'
END_DATE = None  # None = today

# Colors for each status
STATUS_COLORS = {
    'On Time': '#00CC96',  # Teal green
    'Minor': '#FECB52',    # Yellow
    'Major': '#EF553B',    # Red
}

# Commute period highlighting
MORNING_COMMUTE = (6, 9)   # 6:00 - 9:00 AM
EVENING_COMMUTE = (16, 19) # 4:00 - 7:00 PM

# Theme
THEME = {
    'bg_color': '#0d1117',
    'text_color': '#c9d1d9',
    'grid_color': '#21262d',
}

FIGURE_WIDTH = 900
FIGURE_HEIGHT = 500

## Load Data

In [3]:
sys.path.insert(0, os.path.join(project_root, 'src'))
from utils.time_utils import calculate_time_difference, normalize_time
from utils.geo_utils import haversine

def load_and_process_data():
    """Load and process train data."""
    conn = sqlite3.connect(DB_PATH)
    
    # Build date filter
    date_filter = ""
    if START_DATE:
        date_filter += f" AND timestamp >= '{START_DATE}'"
    if END_DATE:
        date_filter += f" AND timestamp <= '{END_DATE}'"
    
    query = f"""
    SELECT trip_id, stop_id, vehicle_lat, vehicle_lon, timestamp
    FROM train_locations
    WHERE 1=1 {date_filter}
    """
    raw_df = pd.read_sql_query(query, conn)
    conn.close()
    
    if raw_df.empty:
        print("No data found")
        return pd.DataFrame()
    
    raw_df['timestamp'] = pd.to_datetime(raw_df['timestamp'], errors='coerce')
    raw_df = raw_df.dropna(subset=['timestamp'])
    print(f"Loaded {len(raw_df):,} records")
    
    # Load GTFS
    stops_df = pd.read_csv(os.path.join(GTFS_PATH, 'stops.txt'))
    stops_df = stops_df[stops_df['stop_id'].str.isnumeric()]
    stop_times_df = pd.read_csv(os.path.join(GTFS_PATH, 'stop_times.txt'))
    
    # Type conversion
    raw_df['stop_id'] = pd.to_numeric(raw_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    raw_df['trip_id'] = pd.to_numeric(raw_df['trip_id'].astype(str), errors='coerce').astype('Int64')
    stops_df['stop_id'] = pd.to_numeric(stops_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    stop_times_df['stop_id'] = pd.to_numeric(stop_times_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    stop_times_df['trip_id'] = pd.to_numeric(stop_times_df['trip_id'].astype(str), errors='coerce').astype('Int64')
    
    raw_df = raw_df.dropna(subset=['trip_id', 'stop_id'])
    
    # Merge
    df = pd.merge(raw_df, stop_times_df[['trip_id', 'stop_id', 'arrival_time']], 
                  on=['trip_id', 'stop_id'], how='inner')
    df = pd.merge(df, stops_df[['stop_id', 'stop_lat', 'stop_lon']], 
                  on='stop_id', how='inner')
    
    # Distance calculation
    df['distance'] = df.apply(lambda r: haversine(
        r['vehicle_lat'], r['vehicle_lon'], r['stop_lat'], r['stop_lon']
    ), axis=1)
    
    df['date'] = df['timestamp'].dt.date
    df['hour'] = df['timestamp'].dt.hour
    df['arrival_time'] = df['arrival_time'].apply(normalize_time)
    
    # Find arrivals
    min_dist = df.groupby(['trip_id', 'stop_id', 'date'])['distance'].min().reset_index()
    merged = pd.merge(df, min_dist, on=['trip_id', 'stop_id', 'date', 'distance'])
    arrivals = merged.groupby(['trip_id', 'stop_id', 'date']).first().reset_index()
    
    # Calculate delay
    arrivals['delay_min'] = arrivals.apply(
        lambda r: calculate_time_difference(r['arrival_time'], r['timestamp']), axis=1
    )
    arrivals.loc[arrivals.delay_min > 500, 'delay_min'] = 0
    arrivals.loc[arrivals.delay_min < -100, 'delay_min'] = 0
    
    # Status
    arrivals['status'] = 'On Time'
    arrivals.loc[(arrivals.delay_min > 4) & (arrivals.delay_min <= 15), 'status'] = 'Minor'
    arrivals.loc[arrivals.delay_min > 15, 'status'] = 'Major'
    
    print(f"Processed {len(arrivals):,} arrivals")
    return arrivals

arrivals = load_and_process_data()
arrivals['status'].value_counts()

Base directory: D:\caltrain-tracker
Dotenv path: D:\caltrain-tracker\.env
Dotenv file exists: True
Loaded 1,526,864 records
Processed 243,075 arrivals


status
On Time    224212
Minor       16035
Major        2828
Name: count, dtype: int64

## Calculate Hourly Statistics

In [4]:
def calculate_hourly_stats(arrivals):
    """Calculate delay distribution by hour."""
    # Group by hour and status
    hourly = arrivals.groupby(['hour', 'status']).size().reset_index(name='count')
    
    # Pivot to get columns for each status
    pivot = hourly.pivot(index='hour', columns='status', values='count').fillna(0)
    
    # Ensure all statuses exist
    for status in ['On Time', 'Minor', 'Major']:
        if status not in pivot.columns:
            pivot[status] = 0
    
    # Reorder columns
    pivot = pivot[['On Time', 'Minor', 'Major']]
    
    # Calculate percentages
    pivot['total'] = pivot.sum(axis=1)
    pivot['on_time_pct'] = (pivot['On Time'] / pivot['total'] * 100).round(1)
    
    # Ensure all hours 0-23 exist
    full_hours = pd.DataFrame({'hour': range(24)})
    pivot = pivot.reset_index()
    pivot = full_hours.merge(pivot, on='hour', how='left').fillna(0)
    
    return pivot

hourly_stats = calculate_hourly_stats(arrivals)
hourly_stats

,hour,On Time,Minor,Major,total,on_time_pct
0,0,38.0,0.0,0.0,38.0,100.0
1,1,0.0,0.0,0.0,0.0,0.0
2,2,0.0,0.0,0.0,0.0,0.0
3,3,210.0,0.0,0.0,210.0,100.0
4,4,623.0,0.0,0.0,623.0,100.0
5,5,5888.0,129.0,3.0,6020.0,97.8
6,6,8893.0,261.0,48.0,9202.0,96.6
7,7,13790.0,647.0,68.0,14505.0,95.1
8,8,14993.0,1059.0,160.0,16212.0,92.5
9,9,13586.0,934.0,131.0,14651.0,92.7


## Create Bar Chart

In [6]:
def create_hourly_chart(hourly_stats):
    """Create stacked bar chart by hour."""
    fig = go.Figure()
    
    # Add stacked bars in order: On Time, Minor, Major
    for status in ['On Time', 'Minor', 'Major']:
        fig.add_trace(go.Bar(
            x=hourly_stats['hour'],
            y=hourly_stats[status],
            name=status,
            marker_color=STATUS_COLORS[status],
            hovertemplate=(
                f'<b>{status}</b><br>'
                'Hour: %{x}:00<br>'
                'Count: %{y:,.0f}<extra></extra>'
            )
        ))
    
    # Add commute period highlighting
    shapes = []
    
    # Morning commute
    shapes.append(dict(
        type='rect',
        xref='x', yref='paper',
        x0=MORNING_COMMUTE[0] - 0.5, x1=MORNING_COMMUTE[1] + 0.5,
        y0=0, y1=1,
        fillcolor='rgba(255, 193, 7, 0.1)',
        line=dict(width=0),
        layer='below'
    ))
    
    # Evening commute
    shapes.append(dict(
        type='rect',
        xref='x', yref='paper',
        x0=EVENING_COMMUTE[0] - 0.5, x1=EVENING_COMMUTE[1] + 0.5,
        y0=0, y1=1,
        fillcolor='rgba(255, 193, 7, 0.1)',
        line=dict(width=0),
        layer='below'
    ))
    
    # Annotations for commute periods
    annotations = [
        dict(
            x=(MORNING_COMMUTE[0] + MORNING_COMMUTE[1]) / 2,
            y=1.02, yref='paper',
            text='Morning Commute',
            showarrow=False,
            font=dict(size=10, color=THEME['text_color']),
        ),
        dict(
            x=(EVENING_COMMUTE[0] + EVENING_COMMUTE[1]) / 2,
            y=1.02, yref='paper',
            text='Evening Commute',
            showarrow=False,
            font=dict(size=10, color=THEME['text_color']),
        ),
    ]
    
    # Calculate summary stats
    total = hourly_stats['total'].sum()
    on_time_total = hourly_stats['On Time'].sum()
    on_time_pct = (on_time_total / total * 100) if total > 0 else 0
    
    fig.update_layout(
        title=dict(
            text=f'🕐 Train Delays by Hour of Day<br><span style="font-size:14px;color:{THEME["text_color"]}">{int(total):,} arrivals | {on_time_pct:.1f}% on-time</span>',
            font=dict(size=22, color=THEME['text_color']),
            x=0.5,
            xanchor='center'
        ),
        barmode='stack',
        xaxis=dict(
            title='Hour of Day',
            tickmode='array',
            tickvals=list(range(24)),
            ticktext=[f'{h}:00' for h in range(24)],
            tickangle=45,
            tickfont=dict(color=THEME['text_color']),
            # titlefont=dict(color=THEME['text_color']),
            gridcolor=THEME['grid_color'],
        ),
        yaxis=dict(
            title='Number of Arrivals',
            tickfont=dict(color=THEME['text_color']),
            # titlefont=dict(color=THEME['text_color']),
            gridcolor=THEME['grid_color'],
        ),
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=-0.25,
            xanchor='center',
            x=0.5,
            font=dict(color=THEME['text_color']),
        ),
        plot_bgcolor=THEME['bg_color'],
        paper_bgcolor=THEME['bg_color'],
        shapes=shapes,
        annotations=annotations,
        height=FIGURE_HEIGHT,
        width=FIGURE_WIDTH,
        margin=dict(l=60, r=40, t=100, b=100),
    )
    
    return fig

fig = create_hourly_chart(hourly_stats)
fig.show()

## Peak Hour Analysis

In [7]:
def analyze_peaks(hourly_stats):
    """Analyze peak hours."""
    print("=" * 50)
    print("📊 HOURLY DELAY ANALYSIS")
    print("=" * 50)
    
    # Busiest hours
    top_hours = hourly_stats.nlargest(5, 'total')
    print("\n🚆 Busiest Hours:")
    for _, row in top_hours.iterrows():
        print(f"  {int(row['hour']):02d}:00 - {int(row['total']):,} arrivals ({row['on_time_pct']:.1f}% on-time)")
    
    # Worst on-time performance
    has_data = hourly_stats[hourly_stats['total'] > 10]  # Filter low-volume hours
    worst = has_data.nsmallest(5, 'on_time_pct')
    print("\n⚠️ Hours with Most Delays:")
    for _, row in worst.iterrows():
        print(f"  {int(row['hour']):02d}:00 - {row['on_time_pct']:.1f}% on-time")
    
    # Best on-time performance
    best = has_data.nlargest(5, 'on_time_pct')
    print("\n✅ Best Hours:")
    for _, row in best.iterrows():
        print(f"  {int(row['hour']):02d}:00 - {row['on_time_pct']:.1f}% on-time")

analyze_peaks(hourly_stats)

📊 HOURLY DELAY ANALYSIS

🚆 Busiest Hours:
  18:00 - 17,182 arrivals (89.3% on-time)
  16:00 - 16,862 arrivals (91.6% on-time)
  17:00 - 16,444 arrivals (88.9% on-time)
  08:00 - 16,212 arrivals (92.5% on-time)
  09:00 - 14,651 arrivals (92.7% on-time)

⚠️ Hours with Most Delays:
  17:00 - 88.9% on-time
  18:00 - 89.3% on-time
  20:00 - 90.0% on-time
  22:00 - 90.1% on-time
  19:00 - 90.4% on-time

✅ Best Hours:
  00:00 - 100.0% on-time
  03:00 - 100.0% on-time
  04:00 - 100.0% on-time
  05:00 - 97.8% on-time
  06:00 - 96.6% on-time


## Export

In [8]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, 'delay_by_hour.html')
fig.write_html(output_path, include_plotlyjs='cdn')
print(f"✅ Exported to: {output_path}")

✅ Exported to: d:\caltrain-tracker\static\plots\delay_by_hour.html
